# WeightLens and CircuitLens on Gemma-2-2B

This is a focused reproduction of feature **layer 0 / index 24** from *Circuit Insights: Towards Interpretability Beyond Activations*. It applies TDHook's public WeightLens projection, CircuitLens attention-decomposition, and circuit-clustering APIs to the published Gemma-2-2B replacement model and Gemma Scope transcoders.

The evidence gates are deliberately narrow: (1) selected WeightLens token IDs/labels must match the published WeightLens record, and (2) selected attention head/source-token contributors must match the first 12 published CircuitLens examples. The bounded clustering run demonstrates the method but is not claimed to reproduce labels fitted on the full 100-example feature artifact.

**Resources.** This notebook downloads the 2B-parameter model and all Gemma transcoders. Allow roughly 12 GB of downloads, 16 GB of accelerator memory, and 20–40 minutes on a recent GPU. It fails closed without CUDA; it is not part of the CPU/network-free documentation suite.


## Environment and immutable evidence

Run from a TDHook checkout with `uv run --extra circuit-lens --extra circuit-clustering --group notebooks jupyter lab`. The checked-in slice contains only the inputs and selected contributors needed for the bounded gate. Its source dataset blobs and the WeightLens/CircuitLens implementation revisions are recorded in the file itself.


In [ ]:
from __future__ import annotations

import importlib.metadata
import json
import random
import subprocess
from pathlib import Path

import numpy as np
import torch
from circuit_tracer import ReplacementModel

from tdhook.attribution import (
    CircuitLensArtifact,
    attention_contributions,
    cluster_circuit_artifacts,
)
from tdhook.weights import analyze_input_invariant_feature, select_projection_outliers

SEED = 127
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True, warn_only=True)

if not torch.cuda.is_available():
    raise RuntimeError("This exact Gemma-2-2B reproduction requires a CUDA accelerator with about 16 GB memory.")

device = torch.device("cuda")
dtype = torch.bfloat16
reference_path = Path("../assets/gemma-2-2b-feature-24-reference.json")
reference = json.loads(reference_path.read_text())
feature_layer = reference["feature"]["layer"]
feature_index = reference["feature"]["index"]
reference["provenance"]

In [ ]:
model = ReplacementModel.from_pretrained(
    reference["provenance"]["model"],
    f"{reference['provenance']['transcoder_set']}@{reference['provenance']['transcoder_revision']}",
    revision=reference["provenance"]["model_revision"],
    device=device,
    dtype=dtype,
)
model.eval()

selected_transcoder = model.transcoders[feature_layer]


def encoder_for_tdhook(transcoder):
    """Normalize circuit-tracer releases to TDHook's [model, feature] convention."""
    weights = transcoder.W_enc
    if weights.shape[0] == model.cfg.d_model:
        return weights
    if weights.shape[1] == model.cfg.d_model:
        return weights.T
    raise ValueError(f"Unrecognized encoder shape: {tuple(weights.shape)}")


# The selected feature is in layer 0, so WeightLens needs no earlier dictionaries.
# Keeping one layer also avoids materializing every lazily loaded decoder.
feature_encoders = (encoder_for_tdhook(selected_transcoder),)
feature_decoders = (selected_transcoder.W_dec,)
token_labels = tuple(model.tokenizer.decode([index]) for index in range(model.cfg.d_vocab))

## WeightLens gate

TDHook projects the selected transcoder encoder into the Gemma embedding dictionary and its decoder into the unembedding dictionary. Candidate pooling and sample-standard-deviation z-scores match the public WeightLens implementation (`k=1000`, token threshold `5.5`). Layer 0 has no earlier transcoder layers, so this gate covers embedding and output-logit candidates only.


In [ ]:
weight_artifact = analyze_input_invariant_feature(
    feature_layer=feature_layer,
    feature_index=feature_index,
    embedding=model.W_E,
    unembedding=model.W_U,
    feature_encoders=feature_encoders,
    feature_decoders=feature_decoders,
    token_outlier_threshold=5.5,
    feature_outlier_threshold=4.0,
    token_pool_size=1000,
    feature_pool_size=100,
    input_token_labels=token_labels,
    output_token_labels=token_labels,
)

expected_positive = [item["token_id"] for item in reference["weightlens"]["embedding_positive"]]
expected_negative = [item["token_id"] for item in reference["weightlens"]["embedding_negative"]]
observed_positive = [item.index for item in weight_artifact.embedding_positive]
observed_negative = [item.index for item in weight_artifact.embedding_negative]
observed_output_labels = [item.label for item in weight_artifact.output_positive]

assert observed_positive == expected_positive, (observed_positive, expected_positive)
assert observed_negative == expected_negative, (observed_negative, expected_negative)
assert observed_output_labels == reference["weightlens"]["output_positive_labels"]

{
    "embedding_positive": [(item.index, item.label, item.score) for item in weight_artifact.embedding_positive],
    "embedding_negative": [(item.index, item.label, item.score) for item in weight_artifact.embedding_negative],
    "output_positive": [(item.index, item.label, item.score) for item in weight_artifact.output_positive],
}

## CircuitLens contributor gate

For each published input, TransformerLens exposes the observed attention pattern and value vectors. TDHook decomposes the layer-0 attention output with the observed pattern frozen and the target feature encoder as the local output gradient. Applying the paper's z-score threshold (`5.0`) must select the same head/source-token identities as the published artifact. Score deltas are reported because CUDA kernels, bfloat16 rounding, and dependency revisions can change the last few bits.


In [ ]:
pattern_name = "blocks.0.attn.hook_pattern"
value_name = "blocks.0.attn.hook_v"
feature_input_name = f"blocks.0.{model.feature_input_hook}"
names = {pattern_name, value_name, feature_input_name}
encoder = feature_encoders[feature_layer][:, feature_index].detach().float().cpu()
output_weight = model.blocks[0].attn.W_O.detach().float().cpu()

circuit_artifacts = []
score_deltas = []
activation_deltas = []

for sample in reference["samples"]:
    tokens = torch.tensor(sample["input_ids"], device=device).unsqueeze(0)
    with torch.no_grad():
        _, cache = model.run_with_cache(tokens, names_filter=lambda name: name in names, stop_at_layer=1)

    pattern = cache[pattern_name][0].detach().float().cpu()
    values = cache[value_name][0].detach().float().cpu()
    if values.shape[1] != pattern.shape[0]:
        values = values.repeat_interleave(pattern.shape[0] // values.shape[1], dim=1)

    position = sample["target_position"]
    output_gradient = torch.zeros(pattern.shape[1], model.cfg.d_model)
    output_gradient[position] = encoder
    all_attention = attention_contributions(
        pattern,
        values,
        output_weight,
        output_gradient,
        layer=0,
        target_position=position,
    )
    selected_positions = select_projection_outliers(
        torch.tensor([item.score for item in all_attention]),
        threshold=5.0,
        largest=True,
    )
    selected = tuple(all_attention[item.index] for item in selected_positions)

    observed_ids = {(item.head_index, item.source_token) for item in selected}
    expected_ids = {(item["head"], item["source_token"]) for item in sample["attention"]}
    assert observed_ids == expected_ids, (position, observed_ids, expected_ids)

    observed_by_id = {(item.head_index, item.source_token): item.score for item in selected}
    score_deltas.extend(
        abs(observed_by_id[(item["head"], item["source_token"])] - item["score"]) for item in sample["attention"]
    )
    feature_input = cache[feature_input_name][0, position].detach()
    target_activation = float(selected_transcoder.encode(feature_input)[feature_index].float().cpu())
    activation_deltas.append(abs(target_activation - sample["target_activation"]))
    circuit_artifacts.append(
        CircuitLensArtifact(
            target_layer=feature_layer,
            target_feature_index=feature_index,
            target_position=(position,),
            target_activation=target_activation,
            upstream_features=(),
            attention=selected,
            output_logits=(),
        )
    )

discrepancies = {
    "max_attention_score_abs_delta": max(score_deltas, default=0.0),
    "max_target_activation_abs_delta": max(activation_deltas, default=0.0),
}
discrepancies

## Bounded circuit clustering

TDHook converts contributors to relative-token circuit signatures, retains contributors appearing in at least 5% of this 12-example slice, and clusters Jaccard distances with DBSCAN. These labels are a bounded demonstration. The published labels were fitted on 100 examples, so label agreement is neither asserted nor interpreted as a reproduction gate.


In [ ]:
clusters = cluster_circuit_artifacts(
    circuit_artifacts,
    min_frequency=0.05,
    min_abs_score=0.0,
    eps=0.8,
    min_samples=2,
)

bounded_cluster_report = {
    "examples": len(circuit_artifacts),
    "labels": clusters.labels,
    "reference_full_sample_labels_for_same_inputs": tuple(item["reference_label"] for item in reference["samples"]),
    "nonempty_signatures": sum(bool(item) for item in clusters.filtered_signatures),
}
bounded_cluster_report

## Provenance and claim boundary

The final record makes the run auditable. Passing assertions support only candidate/contributor agreement for feature 0/24 on this bounded public slice. They do not reproduce every paper table, the 24M-token activation collection, feature descriptions, evaluation scores, or causal intervention effects.


In [ ]:
def package_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


run_record = {
    "model": reference["provenance"]["model"],
    "transcoder_set": reference["provenance"]["transcoder_set"],
    "feature": reference["feature"],
    "sample_count": len(reference["samples"]),
    "seed": SEED,
    "device": str(device),
    "device_name": torch.cuda.get_device_name(device),
    "dtype": str(dtype),
    "torch": torch.__version__,
    "tdhook": package_version("tdhook"),
    "circuit_tracer": package_version("circuit-tracer"),
    "transformer_lens": package_version("transformer-lens"),
    "scikit_learn": package_version("scikit-learn"),
    "tdhook_revision": subprocess.run(
        ["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
    ).stdout.strip(),
    "reference": reference["provenance"],
    "numerical_discrepancies": discrepancies,
    "cluster_report": bounded_cluster_report,
    "limits": [
        "12 of 100 published examples for one feature",
        "layer 0 has no upstream transcoder-feature contributors",
        "no 24M-token activation collection",
        "no paper-wide table, description, evaluation, or causal-effect reproduction",
    ],
}
print(json.dumps(run_record, indent=2, default=list))